In [ ]:
from pyspark import SparkContext

sc = SparkContext("local", "Simple App")

filename = "data/A.txt"
file_A = sc.textFile(filename).cache()

filename = "data/B.txt"
file_B = sc.textFile(filename).cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 10:47:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
file_A = file_A.map(lambda x: int(x.strip()))
file_A.collect()

file_B = file_B.map(lambda x: int(x.strip()))
file_B.collect()

[1, 1, 1, 1, 1, 3, 5, 7, 9, 10, 10, 10, 12, 14, 16, 18, 20]

In [3]:
file_A_set = file_A.distinct()
file_B_set = file_B.distinct()

file_A_set.collect()
file_B_set.collect()

[1, 3, 5, 7, 9, 10, 12, 14, 16, 18, 20]

In [33]:
union = file_A_set.union(file_B_set).distinct()
union.collect()

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 1, 3, 5, 7, 9]

# Q2

In [5]:
intersection = file_A_set.intersection(file_B_set)
intersection.collect()

[10, 12, 14, 1]

# Q3

In [6]:
union = file_A.union(file_B)
union.collect()

[1,
 1,
 2,
 2,
 4,
 6,
 8,
 10,
 10,
 10,
 10,
 12,
 14,
 1,
 1,
 1,
 1,
 1,
 3,
 5,
 7,
 9,
 10,
 10,
 10,
 12,
 14,
 16,
 18,
 20]

# Q4

In [16]:
file_A_dict = file_A.map(lambda x: (x, (1,0)))
file_B_dict = file_B.map(lambda x: (x, (0,1)))

union_dict = file_A_dict.union(file_B_dict).reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))

union_dict.sortByKey().collect()

[(1, (2, 5)),
 (2, (2, 0)),
 (3, (0, 1)),
 (4, (1, 0)),
 (5, (0, 1)),
 (6, (1, 0)),
 (7, (0, 1)),
 (8, (1, 0)),
 (9, (0, 1)),
 (10, (4, 3)),
 (12, (1, 1)),
 (14, (1, 1)),
 (16, (0, 1)),
 (18, (0, 1)),
 (20, (0, 1))]

In [ ]:
def dict_intersection(x):
    min_val = min(x[1][0], x[1][1])

    return min_val * [x[0]]

union_dict.flatMap(dict_intersection).collect()

[10, 10, 10, 12, 14, 1, 1]

# Q5

In [ ]:
def dict_diff(x):
    diff_val = abs(x[1][0] - x[1][1])

    return diff_val * [x[0]]

union_dict.flatMap(dict_diff).collect()

[2, 2, 4, 6, 8, 10, 16, 18, 20, 1, 1, 1, 3, 5, 7, 9]

# Q6


In [20]:
def symmetric_diff(x):
    sym_val = max(x[1][0], x[1][1]) - min(x[1][0], x[1][1])

    return sym_val * [x[0]]

union_dict.flatMap(symmetric_diff).collect()

[2, 2, 4, 6, 8, 10, 16, 18, 20, 1, 1, 1, 3, 5, 7, 9]

# Part 4

## Q1

In [ ]:
filename = "data/M.txt"
file_matrix = sc.textFile(filename).cache()

filename = "data/V.txt"
file_V = sc.textFile(filename).cache()
filename = "data/W.txt"
file_W = sc.textFile(filename).cache()

In [30]:
def format_vector(x):
    x = x.split(' ')
    x_0 = int(x[0])
    x_1 = float(x[1])
    return (x_0, x_1)

V = file_V.map(format_vector)
V.collect()
W = file_W.map(format_vector)
W.collect()

[(1, -1.5), (2, 2.0), (3, 2.3), (4, 2.0), (6, 2.5)]

In [32]:
def format_matrix(x):
    x = x.split(' ')
    x_0 = int(x[0])
    x_1 = int(x[1])
    x_2 = float(x[2])
    return (x_0, x_1, x_2)

M = file_matrix.map(format_matrix)
M.collect()

[(1, 1, 3.2),
 (1, 2, 2.4),
 (1, 3, 7.0),
 (1, 4, 2.0),
 (2, 2, 7.1),
 (2, 3, -1.0),
 (3, 3, 1.0)]

## Q2

In [39]:
from math import sqrt

V_norm = sqrt(V.map(lambda x: x[-1] ** 2).sum())
V_norm

6.159545437773797

## Q3

In [49]:
U = V.union(W).reduceByKey(lambda a,b: a+b)
U.filter(lambda x: x[1] != 0).sortByKey().collect()

[(2, 7.0), (3, 2.3), (4, 3.3), (6, 2.5), (7, 3.0)]